In [1]:
import os, torch

import transformers

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
    get_peft_model,
)

import datasets
from trl import SFTTrainer, setup_chat_format

import json

In [2]:
torch.cuda.is_available()

True

# Constants

In [3]:
MAX_OUTPUT_TOKENS = 128
LR = 1e-4
OUTPUT_DIR = "Pozdnyakov-AI(filtered-dataset)"
NUM_EPOCHS=1

In [4]:
val_dataset_samples = 20

# Processing dataset

In [5]:
dataset_path = r"C:\Users\vital\PycharmProjects\Pozdnyakov-Vlad-AI\dataset\processed_dataset.json"
with open(dataset_path, "r", encoding="utf_8_sig") as json_file:
    dirty_dataset = json.load(json_file)

len(dirty_dataset)

1454

In [6]:
dict_dataset = []

for sample in dirty_dataset:
    if len(sample[1]["content"]) + 3 > MAX_OUTPUT_TOKENS * 2:
        continue
    dict_dataset.append(sample)

len(dict_dataset)

1029

# Unloading required utils

In [7]:
model_name = "t-bank-ai/T-lite-instruct-0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map="auto",
    quantization_config=bnb_config,
    attn_implementation=None
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [8]:
adaptated_dataset = [
    tokenizer.apply_chat_template(current_sample, tokenize=False) 
    for current_sample in dict_dataset
]

column_dataset = {"text": adaptated_dataset[val_dataset_samples:]}
val_column_dataset = {"text": adaptated_dataset[:val_dataset_samples]}
dataset = datasets.Dataset.from_dict(column_dataset)
val_dataset = datasets.Dataset.from_dict(val_column_dataset)

In [9]:
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

def generate(prompt: str) -> str:
    messages = [
        {"role": "user", "content": prompt}
    ]
    
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=False,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
     input_ids,
     max_new_tokens=96,
     eos_token_id=terminators
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)
    

# Getting Training utils

In [10]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['up_proj', 'down_proj', 'gate_proj', 'k_proj', 'q_proj', 'v_proj', 'o_proj']
)

In [11]:
training_arguments = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=NUM_EPOCHS,
    eval_strategy="steps",
    eval_steps=0.05,
    warmup_steps=10,
    learning_rate=LR,
    fp16=True,
    group_by_length=True,
    save_steps=50,
    report_to=[],
)

In [12]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    max_seq_length=MAX_OUTPUT_TOKENS,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False,
)

C:\Users\vital\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\utils\_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
C:\Users\vital\AppData\Local\Programs\Python\Python312\Lib\site-packages\trl\trainer\sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
C:\Users\vital\AppData\Local\Programs\Python\Python312\Lib\site-packages\trl\trainer\sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/1009 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

C:\Users\vital\AppData\Local\Programs\Python\Python312\Lib\site-packages\trl\trainer\sft_trainer.py:412: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [13]:
trainer.train()

Step,Training Loss,Validation Loss
26,No log,2.526369
52,No log,2.432946
78,No log,2.378201
104,No log,2.325752
130,No log,2.302122
156,No log,2.312129
182,No log,2.247852
208,No log,2.252222
234,No log,2.227012
260,No log,2.222920


TrainOutput(global_step=505, training_loss=2.028029171783145, metrics={'train_runtime': 561.9648, 'train_samples_per_second': 1.795, 'train_steps_per_second': 0.899, 'total_flos': 4198842531840000.0, 'train_loss': 2.028029171783145, 'epoch': 1.0})